# TP3 - LayerNorm, MLP et connexions residuelles

La normalisation et le feed-forward transforment chaque token independamment, tandis que le residuel preserve le chemin de l'information.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(1337)

class LayerNorm1d(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        xhat = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * xhat + self.beta

x = torch.randn(32, 100)
ours = LayerNorm1d(100)
reference = nn.LayerNorm(100)
reference.weight.data.copy_(ours.gamma.data)
reference.bias.data.copy_(ours.beta.data)
assert torch.allclose(ours(x), reference(x), atol=1e-6)
y = ours(x)
print('feature 0 mean/std (not constrained):', y[:, 0].mean().item(), y[:, 0].std().item())
print('token 0 mean/std:', y[0].mean().item(), y[0].std().item())


La normalisation se fait sur la derniere dimension: les features d'un token. `eps` evite une division instable; `gamma` et `beta` permettent au modele d'apprendre l'echelle et le decalage.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, multiplier=4, activation=nn.GELU):
        super().__init__()
        hidden = multiplier * n_embd
        self.net = nn.Sequential(nn.Linear(n_embd, hidden), activation(), nn.Linear(hidden, n_embd))

    def forward(self, x):
        return self.net(x)

B, T, C = 4, 8, 100
ffwd = FeedForward(C)
delta = ffwd(torch.randn(B, T, C))
print('delta:', delta.shape)
print('parametres C-4C-C:', sum(p.numel() for p in ffwd.parameters()))
for multiplier in (2, 4, 8):
    variant = FeedForward(C, multiplier)
    print(f'C-{multiplier}C-C:', sum(p.numel() for p in variant.parameters()))


In [ ]:
n_embd = 64
tokens = torch.randn(B, T, n_embd)
ln = nn.LayerNorm(n_embd)
ffwd = FeedForward(n_embd)
delta = ffwd(ln(tokens))
with_residual = tokens + delta
without_residual = delta
print(tokens.shape, delta.shape, with_residual.shape, without_residual.shape)
assert with_residual.shape == tokens.shape


### Questions et reponses

1. **Sur quel axe normaliser ?** Sur la derniere dimension, celle des `C` features de chaque token. Pour `x` de forme `(B,T,C)`, on utilise `dim=-1`; les positions et les exemples restent independants.
2. **A quoi servent `eps`, `gamma` et `beta` ?** `eps` evite une division par zero, `gamma` apprend une echelle par feature et `beta` apprend un decalage par feature. La variance utilisee est la variance de population (`unbiased=False`), comme `nn.LayerNorm`.
3. **Quelle ligne doit etre proche de `(0,1)` ?** `y[0, :]`, car elle regarde les features d'un seul token. `y[:, 0]` traverse les tokens et n'est donc pas contrainte a avoir moyenne nulle et ecart-type un.
4. **Le MLP melange-t-il les tokens ?** Non. Les couches lineaires operent sur la derniere dimension et la meme transformation est appliquee independamment a chaque position. Elles transforment `C -> 4C -> C`.
5. **Pourquoi revenir a `C` ?** Pour que la mise a jour `x + delta` soit definie et que le MLP s'insere dans le bloc sans changer la largeur du residuel.
6. **Quel est l'effet du residuel ?** `x + delta` conserve le signal d'origine et laisse la sous-couche apprendre une correction; sans addition, seule la transformation du MLP est transmise. ReLU/GELU et les largeurs `2C`, `4C`, `8C` modifient la capacite et le nombre de parametres, pas la forme externe.